# Final Evaluation, Hypothesis Decisions, and Project Synthesis

**SoftUni Deep Learning Final Project**  
**Notebook:** `09_final_evaluation.ipynb`

This notebook consumes the canonical result packages exported by Notebooks 04–08. It does not retrain models. Its role is to validate artifact completeness, consolidate comparable results, evaluate the six predefined hypotheses, and export the evidence required for the final written report.

Static normalized-price models and contract-level Longstaff–Schwartz methods are reported separately because they operate on different evaluation units and price scales.


## 1. Environment and strict final-evaluation configuration

`STRICT_MODE = True` is the submission setting. Missing or schema-incompatible required artifacts stop execution instead of producing polished but empty placeholder files.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == "notebooks"
    else NOTEBOOK_DIR
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.artifact_registry import audit_artifacts
from src.evaluation.final_project_evaluation import (
    build_boundary_comparison,
    build_financial_consistency_table,
    build_hypothesis_evidence,
    build_literature_handoff,
    build_lsm_comparison,
    build_metric_inventory,
    build_ood_comparison,
    build_runtime_comparison_from_results,
    build_static_ablation_table,
    build_static_pricing_table,
    load_project_results,
)
from src.evaluation.final_reporting import (
    export_final_evaluation,
    write_json,
)
from src.evaluation.hypothesis_testing import (
    decide_all_hypotheses,
)

STRICT_MODE = True
FINAL_OUTPUT_DIR = (
    PROJECT_ROOT / "artifacts" / "final_evaluation"
)
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DESIGN_PATH = (
    PROJECT_ROOT
    / "data"
    / "manifests"
    / "final_evaluation_design.json"
)

design = (
    json.loads(DESIGN_PATH.read_text(encoding="utf-8"))
    if DESIGN_PATH.is_file()
    else {
        "expected_total_observations": 1_450_000,
        "artifact_version": "canonical_final_evaluation_v2",
    }
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Strict mode:  {STRICT_MODE}")
print(f"Output dir:   {FINAL_OUTPUT_DIR}")
display(pd.Series(design, name="value").to_frame())


## 2. Canonical artifact audit

The audit validates exact files, JSON keys, table columns, row presence, completion status, and canonical selected checkpoints. Notebook 09's own previous exports are deliberately excluded from the upstream registry.


In [ ]:
artifact_audit = audit_artifacts(PROJECT_ROOT)

display(
    artifact_audit[
        [
            "name",
            "category",
            "required_for_final",
            "found",
            "valid",
            "resolved_path",
            "notes",
        ]
    ]
)

required_audit = artifact_audit.loc[
    artifact_audit["required_for_final"]
].copy()

invalid_required = required_audit.loc[
    ~required_audit["valid"]
].copy()

valid_required = int(required_audit["valid"].sum())
total_required = int(len(required_audit))

print(
    f"Valid required artifacts: "
    f"{valid_required}/{total_required}"
)

if STRICT_MODE and not invalid_required.empty:
    missing_summary = "\n".join(
        f"- {row.name}: {row.notes}"
        for row in invalid_required[
            ["name", "notes"]
        ].itertuples(index=False)
    )
    raise RuntimeError(
        "Final evaluation cannot continue because required "
        "artifacts are missing or invalid:\n"
        + missing_summary
        + "\nRun the canonical export cells in Notebooks "
        "04–08 or execute `dvc pull`."
    )


## 3. Load canonical packages and build the metric inventory

Every model family has an explicit adapter. Metrics are no longer inferred by searching for any key that happens to end in `mae` or `rmse`.


In [ ]:
project_results = load_project_results(PROJECT_ROOT)
metric_inventory = build_metric_inventory(PROJECT_ROOT)

package_status = pd.DataFrame(
    [
        {
            "package": name,
            "loaded": payload is not None,
            "payload_type": type(payload).__name__,
        }
        for name, payload in project_results.items()
    ]
)

display(package_status)
display(metric_inventory.head(40))
print(
    f"Metric inventory rows: {len(metric_inventory):,}"
)


## 4. Consolidated in-domain static pricing comparison

This table contains only models evaluated on the common static normalized-price test split. Lower MAE and RMSE are better. Longstaff–Schwartz results are excluded here because they use contract-level raw prices and Monte Carlo confidence intervals.


In [ ]:
static_pricing = build_static_pricing_table(
    project_results
)

if STRICT_MODE and static_pricing.empty:
    raise RuntimeError(
        "Static pricing table is empty."
    )

display(static_pricing)

best_static = static_pricing.loc[
    static_pricing["mae"].idxmin()
]
print(
    "Lowest static test MAE: "
    f"{best_static['model']} "
    f"({best_static['mae']:.8f})"
)


## 5. Financial-consistency comparison

The table combines empirical lower-bound violations and integrated-model internal-consistency diagnostics. Architectural guarantees are distinguished from empirical monotonicity checks.


In [ ]:
financial_consistency = (
    build_financial_consistency_table(
        project_results
    )
)

if STRICT_MODE and financial_consistency.empty:
    raise RuntimeError(
        "Financial-consistency table is empty."
    )

display(financial_consistency)


## 6. Exercise-boundary and classification comparison

Notebook 06 provides the standalone and multi-task boundary comparison. Notebook 08 adds the final integrated exercise head and the corrected boundary-band analysis. F1 and balanced accuracy are left undefined for bands containing only one actual class.


In [ ]:
boundary_comparison = build_boundary_comparison(
    project_results
)

if STRICT_MODE and boundary_comparison.empty:
    raise RuntimeError(
        "Boundary comparison is empty."
    )

display(boundary_comparison)


## 7. Out-of-domain deterioration

OOD MAE is compared with each model's own in-domain MAE. The aggregate hypothesis decision is based on one selected static model, while the full table preserves regime-specific deterioration.


In [ ]:
ood_results = build_ood_comparison(
    project_results,
    static_pricing,
)

if STRICT_MODE and ood_results.empty:
    raise RuntimeError(
        "OOD comparison is empty."
    )

display(ood_results)

ood_summary = (
    ood_results.groupby("model", as_index=False)
    .agg(
        regimes=("regime", "nunique"),
        mean_ood_mae=("ood_mae", "mean"),
        mean_deterioration=(
            "ood_deterioration",
            "mean",
        ),
    )
    .sort_values("mean_ood_mae")
)

display(ood_summary)


## 8. Runtime and computational efficiency

Marginal static inference, per-contract numerical valuation, and one-time training cost are separate cost categories. They are not blended into one misleading speed ranking.


In [ ]:
runtime_comparison = (
    build_runtime_comparison_from_results(
        project_results
    )
)

if STRICT_MODE and runtime_comparison.empty:
    raise RuntimeError(
        "Runtime comparison is empty."
    )

display(runtime_comparison)


## 9. Static-model ablation

The ablation table links model structure with observed test MAE, exercise quality, and available financial-consistency evidence.


In [ ]:
static_ablation = build_static_ablation_table(
    static_pricing,
    financial_consistency,
    boundary_comparison,
)

if STRICT_MODE and static_ablation.empty:
    raise RuntimeError(
        "Static ablation table is empty."
    )

display(static_ablation)


## 10. Classical and neural Longstaff–Schwartz

These results remain separate from the static table. MAE and RMSE are measured against CRR contract prices, while confidence-interval coverage, policy behavior, training cost, and valuation runtime provide additional evidence.


In [ ]:
lsm_comparison = build_lsm_comparison(
    project_results
)

if STRICT_MODE and lsm_comparison.empty:
    raise RuntimeError(
        "LSM comparison is empty."
    )

display(lsm_comparison)


## 11. Automatically derived H1–H6 evidence and decisions

The evidence dictionary is generated from the consolidated tables and canonical packages. No manually edited intermediary JSON file is required.


In [ ]:
hypothesis_evidence = build_hypothesis_evidence(
    project_results,
    static_pricing,
    financial_consistency,
    ood_results,
    runtime_comparison,
)

write_json(
    FINAL_OUTPUT_DIR / "hypothesis_evidence.json",
    hypothesis_evidence,
)

hypothesis_decisions = decide_all_hypotheses(
    hypothesis_evidence
)

display(
    pd.Series(
        hypothesis_evidence,
        name="value",
    ).to_frame()
)
display(hypothesis_decisions)

if (
    STRICT_MODE
    and hypothesis_decisions[
        "decision"
    ].eq("Inconclusive").any()
):
    unresolved = hypothesis_decisions.loc[
        hypothesis_decisions[
            "decision"
        ].eq("Inconclusive"),
        "hypothesis",
    ].tolist()
    raise RuntimeError(
        "Strict final evaluation has unresolved "
        f"hypotheses: {unresolved}"
    )


## 12. Literature-synthesis handoff

The code does not invent citation links. It exports each empirical finding with an explicit manual citation-mapping status so the final written report can connect the findings to the supplied papers.


In [ ]:
literature_synthesis = (
    build_literature_handoff(
        hypothesis_decisions
    )
)

display(literature_synthesis)


## 13. Export final evaluation

Strict export rejects missing required artifacts, empty tables, all-missing tables, and inconclusive hypotheses. A successful run therefore produces a defensible final evidence package rather than a skeleton.


In [ ]:
tables_to_export = {
    "consolidated_model_metrics": static_pricing,
    "metric_inventory": metric_inventory,
    "financial_consistency_table": (
        financial_consistency
    ),
    "boundary_comparison": boundary_comparison,
    "ood_results": ood_results,
    "ood_model_summary": ood_summary,
    "runtime_comparison": runtime_comparison,
    "static_model_ablation": static_ablation,
    "lsm_comparison": lsm_comparison,
    "literature_synthesis": literature_synthesis,
}

status = "READY_FOR_FINAL_WRITEUP"

exported_paths = export_final_evaluation(
    FINAL_OUTPUT_DIR,
    tables=tables_to_export,
    hypothesis_decisions=hypothesis_decisions,
    artifact_audit=artifact_audit,
    summary={
        "status": status,
        "valid_required_artifacts": (
            valid_required
        ),
        "required_artifacts": total_required,
        "expected_total_observations": (
            design.get(
                "expected_total_observations"
            )
        ),
        "best_static_model": (
            best_static["model"]
        ),
        "best_static_mae": (
            best_static["mae"]
        ),
    },
    strict=STRICT_MODE,
)

display(
    pd.Series(
        exported_paths,
        name="path",
    ).to_frame()
)


# Final project conclusion

A successful strict execution establishes that:

- every required upstream artifact is present and schema-compatible;
- static-model results are compared on one common normalized-price test split;
- financial consistency, exercise behavior, OOD deterioration, and runtime are populated from explicit source packages;
- classical and neural Longstaff–Schwartz results are reported on their own contract-level basis;
- H1–H6 decisions are derived automatically from the exported evidence;
- the final write-up package contains no empty placeholder tables.

The remaining work is interpretive writing and manual mapping of the supplied research papers to the empirical findings.
